In [6]:
import pandas as pd

In [7]:
order_level = pd.read_parquet("data/processed/order_level.parquet")
item_level = pd.read_parquet("data/processed/item_level.parquet")
rfm = pd.read_parquet("data/processed/rfm.parquet")

# Get each customer's first order only
first_orders = (
    order_level.sort_values("order_purchase_timestamp")
    .drop_duplicates(subset="customer_unique_id", keep="first")
)

In [8]:
# Label repeat vs one-time using frequency from RFM
first_orders = first_orders.merge(
    rfm[["customer_unique_id", "frequency"]], on="customer_unique_id", how="left"
)
first_orders["is_repeat"] = first_orders["frequency"] >= 2

print(first_orders["is_repeat"].value_counts(normalize=True) * 100)

is_repeat
False    96.974429
True      3.025571
Name: proportion, dtype: float64


In [9]:
# 1. Late delivery on first order
compare_late = first_orders.groupby("is_repeat")["is_late"].mean()

# 2. Review score on first order
compare_review = first_orders.groupby("is_repeat")["review_score"].mean()

# 3. Freight share (freight / (price + freight)) — need item-level or precomputed ratio
first_orders["freight_share"] = first_orders["total_freight"] / (
    first_orders["total_freight"] + first_orders["total_price"]
)
compare_freight = first_orders.groupby("is_repeat")["freight_share"].mean()

# 4. Payment installments
compare_installments = first_orders.groupby("is_repeat")["n_payment_installments"].mean()

# 5. Delivery time
compare_delivery_time = first_orders.groupby("is_repeat")["delivery_time"].mean()

# 6. Customer state (top states per group)
compare_state = (
    first_orders.groupby(["is_repeat", "customer_state"]).size()
    .groupby(level=0).apply(lambda x: x / x.sum() * 100)
    .unstack(level=0)
)

print("Late rate:\n", compare_late)
print("\nReview score:\n", compare_review)
print("\nFreight share:\n", compare_freight)
print("\nInstallments:\n", compare_installments)
print("\nDelivery time:\n", compare_delivery_time)

Late rate:
 is_repeat
False    0.066852
True     0.056114
Name: is_late, dtype: float64

Review score:
 is_repeat
False    4.100500
True     4.148549
Name: review_score, dtype: float64

Freight share:
 is_repeat
False    0.208232
True     0.217641
Name: freight_share, dtype: float64

Installments:
 is_repeat
False    2.906465
True     3.322480
Name: n_payment_installments, dtype: float64

Delivery time:
 is_repeat
False    12.109127
True     11.961702
Name: delivery_time, dtype: float64


#### Repeat vs One-Time: First-Order Comparison

- **Late rate**: repeat 5.6% vs one-time 6.7% — repeat customers had slightly *fewer* late deliveries on their first order
- **Review score**: repeat 4.15 vs one-time 4.10 — small gap, repeat buyers had marginally better first-order experience
- **Freight share**: repeat 21.8% vs one-time 20.8% — repeat buyers paid a slightly higher freight-to-price ratio, not lower
- **Installments**: repeat 3.32 vs one-time 2.91 — repeat buyers used more installments on their first order
- **Delivery time**: repeat 11.96 days vs one-time 12.11 days — essentially the same

**Overall: none of these gaps look large or decisive.** All differences are small (a few % or a few tenths of a point), which suggests **first-order logistics quality is not the main driver of repeat behavior** — repeat buyers didn't get meaningfully faster/cheaper/better-reviewed first orders.

In [10]:
first_order_ids = first_orders["order_id"]
first_order_items = item_level[item_level["order_id"].isin(first_order_ids)]

first_order_items = first_order_items.merge(
    first_orders[["order_id", "is_repeat"]], on="order_id", how="left"
)

category_repeat_rate = (
    first_order_items.groupby("product_category_name_english")["is_repeat"]
    .agg(["mean", "count"])
    .query("count >= 50")  # avoid noisy small categories
    .sort_values("mean", ascending=False)
)

print(category_repeat_rate.head(10))
print(category_repeat_rate.tail(10))

                                   mean  count
product_category_name_english                 
home_appliances                0.088953    697
furniture_bedroom              0.087379    103
fashion_bags_accessories       0.061506   1886
fashion_male_clothing          0.056000    125
fashion_shoes                  0.055118    254
furniture_living_room          0.054054    481
furniture_decor                0.050139   7918
bed_bath_table                 0.048845  10523
food                           0.043121    487
drinks                         0.042857    350
                                           mean  count
product_category_name_english                         
dvds_blu_ray                           0.016393     61
electronics                            0.016159   2723
musical_instruments                    0.014993    667
books_technical                        0.011364    264
construction_tools_lights              0.010169    295
computers                              0.010050    

#### Category Signal
- **Highest repeat rate**: `home_appliances` (8.9%), `furniture_bedroom` (8.7%), `fashion_bags_accessories` (6.2%), `fashion_shoes` (5.5%) — home/furniture and fashion categories show up to ~3x the platform average repeat rate (3.0%)
- **Lowest repeat rate**: `small_appliances_home_oven_and_coffee`, `tablets_printing_image`, `costruction_tools_tools` (all ~0%), `electronics` (1.6%), `musical_instruments` (1.5%) — durable/one-off purchase categories show almost no repeat behavior

**Insight**: repeat purchasing looks driven more by **product category type** (recurring-need categories like home goods/fashion) than by **first-order service quality** (delivery speed, review score were nearly identical between groups). This reframes the retention strategy — win-back campaigns may work better when targeted at customers whose first purchase was in a naturally repeat-friendly category, rather than assuming service improvements alone will drive repeat purchases.

In [11]:
from statsmodels.stats.proportion import proportion_confint

# Combine top and bottom categories into one table for CI check
categories_to_check = pd.concat([category_repeat_rate.head(10), category_repeat_rate.tail(10)])

categories_to_check["successes"] = (categories_to_check["mean"] * categories_to_check["count"]).round().astype(int)

ci_low, ci_high = proportion_confint(
    categories_to_check["successes"], categories_to_check["count"], method="wilson"
)
categories_to_check["ci_low"] = ci_low
categories_to_check["ci_high"] = ci_high

print(categories_to_check[["mean", "count", "ci_low", "ci_high"]])

                                           mean  count    ci_low   ci_high
product_category_name_english                                             
home_appliances                        0.088953    697  0.070010  0.112402
furniture_bedroom                      0.087379    103  0.046651  0.157777
fashion_bags_accessories               0.061506   1886  0.051529  0.073266
fashion_male_clothing                  0.056000    125  0.027388  0.111088
fashion_shoes                          0.055118    254  0.033113  0.090379
furniture_living_room                  0.054054    481  0.037152  0.078023
furniture_decor                        0.050139   7918  0.045546  0.055168
bed_bath_table                         0.048845  10523  0.044889  0.053131
food                                   0.043121    487  0.028374  0.065019
drinks                                 0.042857    350  0.026141  0.069499
dvds_blu_ray                           0.016393     61  0.002900  0.087189
electronics              

#### Assessment: Confidence Interval Check

**Reliable "high repeat rate" categories** (CI clears above the 3.03% baseline, adequate sample size):
- home_appliances (8.9%, CI 7.0–11.2%)
- fashion_bags_accessories (6.2%, CI 5.2–7.3%, n=1,886)
- furniture_decor (5.0%, CI 4.6–5.5%, n=7,918)
- bed_bath_table (4.9%, CI 4.5–5.3%, n=10,523)

**Weaker signal but still above baseline** (wider CI due to smaller n): furniture_bedroom, fashion_shoes, furniture_living_room.

**Not statistically distinguishable from baseline**: fashion_male_clothing, food, drinks — CI overlaps the baseline.

**Reliable "low repeat rate" categories**:
- electronics (1.6%, CI 1.2–2.2%, n=2,723)
- musical_instruments (1.5%, CI 0.8–2.7%)

**Key correction**: the three categories that showed 0% repeat rate (costruction_tools_tools, tablets_printing_image, small_appliances_home_oven_and_coffee) all have CIs that overlap the baseline (up to 3.7–4.9%), due to very small sample sizes (n=75–99). These should **not** be reported as genuinely low-repeat categories — the 0% is sampling noise, not a real effect.

#### Updated Conclusion
- Categories with a **reliably higher** repeat rate: home_appliances, fashion_bags_accessories, furniture_decor, bed_bath_table — all backed by adequate sample sizes and CIs clearly separated from baseline.
- Categories with a **reliably lower** repeat rate: electronics, musical_instruments — the strongest evidence for "durable, one-off purchase" categories seeing less repeat behavior.
- The earlier "0% repeat" categories should be dropped from the final report — sample too small to draw a conclusion.